<a href="https://colab.research.google.com/github/sadiyatamanna/EdvergenceX-Learning/blob/main/Adaptive_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CELL 1: Install Dependencies


In [1]:
!pip install faiss-cpu openai nltk numpy requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.2 MB/s eta 0:00:00


CELL 2: Imports

In [2]:
import numpy as np
import faiss
import nltk
from nltk.tokenize import sent_tokenize
from openai import OpenAI
from google.colab import userdata
nltk.download("punkt")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

CELL 3: Load API Credentials from Colab

In [3]:
API_KEY = userdata.get("API_KEY")
BASE_URL = userdata.get("BASE_URL")

if not API_KEY or not BASE_URL:
    raise RuntimeError("❌ API_KEY / BASE_URL missing in Colab Secrets")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("✅ API loaded successfully")


✅ API loaded successfully


CELL 4: Load External Document

In [4]:
from google.colab import files

# Upload document
uploaded = files.upload()

# Get uploaded filename automatically
filename = list(uploaded.keys())[0]

# Read the document
with open(filename, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Uploaded file:", filename)
print("📄 Document length:", len(raw_text))
print("\nPreview:")
print(raw_text[:500])


Saving Sample.txt to Sample.txt
Uploaded file: Sample.txt
📄 Document length: 1532

Preview:
This project presents a real-time sign language recognition and learning system that helps bridge communication between hearing-impaired individuals and non-signers. Using a webcam, computer vision, and machine learning, the system detects hand landmarks through MediaPipe and classifies gestures using Random Forest for static signs and LSTM for dynamic movements. Recognized gestures are displayed as text and converted into speech, while a learning module allows users to practice and understand s


CELL 5: Semantic Chunking

In [6]:
nltk.download("punkt_tab")
def semantic_chunking(text, max_sentences=5):
    sentences = sent_tokenize(text)
    chunks = []

    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i + max_sentences])
        chunks.append(chunk)

    return chunks

documents = semantic_chunking(raw_text)
print("🧩 Total semantic chunks:", len(documents))


🧩 Total semantic chunks: 2


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


CELL 6: API-Based Embeddings

In [7]:
def embed_texts(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([item.embedding for item in response.data], dtype="float32")

doc_embeddings = embed_texts(documents)
print("📐 Embedding shape:", doc_embeddings.shape)


📐 Embedding shape: (2, 1536)


In [8]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print("📦 Indexed chunks:", index.ntotal)


📦 Indexed chunks: 2


Build Vector Database

In [9]:
def is_related_to_index(query, threshold=1.8):
    q_emb = embed_texts([query])
    distances, _ = index.search(q_emb, 1)
    return distances[0][0] < threshold


def retrieve(query, k=3):
    q_emb = embed_texts([query])
    _, idx = index.search(q_emb, k)
    return [documents[i] for i in idx[0]]


Is Query Related to Index?

In [10]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "Answer strictly using the provided context only.,IF CONTENT IS NOT IN CONTEXT MEANS DONT PROVIDE"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content


In [11]:
def hallucinated(answer, chunks, threshold=0.6):
    texts = chunks + [answer]
    embeddings = embed_texts(texts)

    answer_emb = embeddings[-1]
    chunk_embs = embeddings[:-1]

    similarities = [
        np.dot(answer_emb, c) / (np.linalg.norm(answer_emb) * np.linalg.norm(c))
        for c in chunk_embs
    ]

    return max(similarities) < threshold


In [12]:
def preprocess_query(query):
    if len(query.split()) < 3 and not query.lower().startswith("what is"):
        return f"What is {query}?"
    return query


def rewrite_query(query):
    return f"Explain clearly with examples: {query}"


In [13]:
def web_search(query):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a web-enabled assistant. Provide factual answers."
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


In [14]:
def adaptive_rag(query, max_loops=3):
    original_query = query

    for loop in range(1, max_loops + 1):
        print(f"\n🔁 LOOP {loop}: {query}")

        processed_query = preprocess_query(query)

        if not is_related_to_index(processed_query):
            print("❌ Not related to index → Web search fallback")
            return web_search(processed_query)

        docs = retrieve(processed_query)
        context = " ".join(docs)

        prompt = f"""
Use ONLY the context below.

Context:
{context}

Question:
{query}

Answer:
"""

        answer = generate_answer(prompt)

        if not hallucinated(answer, docs):
            print("✅ Answer verified")
            return answer

        print("⚠️ Hallucination detected → rewriting query")
        query = rewrite_query(original_query)

    return "⚠️ Unable to verify answer after multiple attempts."


In [16]:
result = adaptive_rag("what is Adaptive Rag?")
print("\n🟢 FINAL ANSWER:\n", result)



🔁 LOOP 1: what is Adaptive Rag?
⚠️ Hallucination detected → rewriting query

🔁 LOOP 2: Explain clearly with examples: what is Adaptive Rag?
⚠️ Hallucination detected → rewriting query

🔁 LOOP 3: Explain clearly with examples: what is Adaptive Rag?
⚠️ Hallucination detected → rewriting query

🟢 FINAL ANSWER:
 ⚠️ Unable to verify answer after multiple attempts.


In [17]:
result = adaptive_rag("what is the document all about?")
print("\n🟢 FINAL ANSWER:\n", result)



🔁 LOOP 1: what is the document all about?
✅ Answer verified

🟢 FINAL ANSWER:
 The document is about a real-time sign language recognition and learning system that uses computer vision and machine learning to facilitate communication between hearing-impaired individuals and non-signers. It details the system's features, current capabilities, and future improvements aimed at enhancing accessibility, inclusivity, and effectiveness through support for multiple sign languages, mobile and offline use, handling complex gestures, and personalized learning.
